# Dikeri Ivant Notebook

Objective: Memprediksi penjualan tiket bioskop harian D4-D10 untuk setiap pasangan film dan bioskop berdasarkan penjualan D1-D3 pasangan tersebut, dengan MASE sebagai metrik evaluasi.

# Sinkronisasi versi modules dan libraries

In [41]:
import sys
print(sys.version)
%pip install -q numpy==2.4.4 pandas==2.3.3 scipy==1.17.1 scikit-learn==1.8.0 lightgbm==4.6.0 torch==2.11.0 tqdm==4.67.3 joblib==1.5.3 huggingface_hub==1.24.0

3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\leouw\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Import modules dan libraries

In [42]:
import os, glob, random, re
import numpy as np
import pandas as pd
import torch
import joblib
from tqdm.auto import tqdm
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
from huggingface_hub import snapshot_download
import sdm
from sdm import TableTensor

# Import datasets

In [43]:
_DATA_DIR_CANDIDATES = [
    "../data", "data", "../../data",
    *glob.glob("/kaggle/input/*"), *glob.glob("/kaggle/input/**/", recursive=True),
]
DATA_DIR = next((d for d in _DATA_DIR_CANDIDATES if d and os.path.exists(os.path.join(d, "test_history.csv"))), None)
if DATA_DIR is None:
    raise FileNotFoundError("test_history.csv not found in any candidate DATA_DIR - set it manually.")
print("DATA_DIR =", os.path.abspath(DATA_DIR))

train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date_show"])
test_hist = pd.read_csv(f"{DATA_DIR}/test_history.csv", parse_dates=["date_show"])
test = pd.read_csv(f"{DATA_DIR}/test.csv", parse_dates=["date_show"])
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
movies = pd.read_csv(f"{DATA_DIR}/movies.csv")
holidays = pd.read_csv(f"{DATA_DIR}/holidays.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA_DIR}/ticket_prices.csv")

DATA_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\data


### External datasets
Kalender publik yang tidak ada pada file holidays bawaan: hari cuti bersama, tanggal Ramadan, libur sekolah, dan libur nasional tambahan, ditambah tabel profil film dan chart box office nasional. Semuanya dipublikasikan pada atau sebelum 2025-09-30, sehingga tidak ada data dari periode test yang dipakai.

In [44]:
_EXTERNAL_DIR_CANDIDATES = ["../external", "external", "../../external", *glob.glob("/kaggle/input/*")]
EXTERNAL_DIR = next((d for d in _EXTERNAL_DIR_CANDIDATES if os.path.exists(os.path.join(d, "cuti_bersama.csv"))), None)
if EXTERNAL_DIR is None:
    raise FileNotFoundError("cuti_bersama.csv not found in any candidate EXTERNAL_DIR - set it manually.")
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

cuti = pd.read_csv(f"{EXTERNAL_DIR}/cuti_bersama.csv", parse_dates=["date"])
ramadan = pd.read_csv(f"{EXTERNAL_DIR}/ramadan.csv", parse_dates=["date"])
school = pd.read_csv(f"{EXTERNAL_DIR}/school_holidays.csv", parse_dates=["date"])
extra_holidays = pd.read_csv(f"{EXTERNAL_DIR}/extra_national_holidays.csv", parse_dates=["date"])

EXTERNAL_DIR = c:\MyFolder\Coding\Data Science\Lomba\JOINTS\JOINTS\external


# EDA

### Jumlah baris per hari
Menghitung jumlah baris train.csv per hari. Hari dengan baris jauh lebih sedikit dari hari-hari di sekitarnya adalah rekaman parsial: penjualannya tercatat terlalu rendah, jadi dihapus nanti agar tidak dipelajari sebagai permintaan yang rendah.

In [45]:
rows_per_day = train.groupby("date_show").size()
rows_per_day.nsmallest(10)

date_show
2025-06-09      1
2025-06-07     10
2025-06-11    136
2025-06-14    169
2025-06-15    326
2025-06-16    404
2025-06-10    449
2025-09-09    563
2025-09-08    572
2025-09-07    581
dtype: int64

### Periode data
Membandingkan rentang tanggal train, test_history, dan test. Periode test dimulai setelah periode train, sehingga model harus memprediksi musim yang belum pernah dilihat.

In [46]:
pd.DataFrame({name: [df["date_show"].min().date(), df["date_show"].max().date(), len(df)]
              for name, df in {"train": train, "test_history": test_hist, "test": test}.items()},
             index=["first day", "last day", "rows"])

,train,test_history,test
first day,2025-04-01,2025-10-01,2025-10-04
last day,2025-09-30,2026-03-20,2026-03-27
rows,138959,32323,72611


### Tiket per hari dalam seminggu
Rata-rata tiket per film dan bioskop pada tiap hari dalam seminggu. Akhir pekan jauh lebih ramai dari hari kerja, sehingga hari dari tanggal target diberikan ke model.

In [47]:
train.groupby(train["date_show"].dt.dayofweek)["total_ticket"].mean().round(1).rename("mean tickets (0 = Monday)")

date_show
0    294.7
1    260.9
2    321.0
3    339.3
4    368.4
5    465.4
6    426.6
Name: mean tickets (0 = Monday), dtype: float64

# Aggregate Dataset
Mengubah catatan harian menjadi satu baris per pasangan film dan bioskop untuk setiap hari target, dengan format yang sama untuk train dan test.

### Penentuan D1
D1 adalah hari pertama film tayang di setidaknya setengah dari jumlah bioskop pada rilis terluasnya, sehingga preview dan tayang terbatas tidak menentukan D1. Film yang sudah tayang sebelum periode train dimulai, atau yang D10-nya jatuh setelah periode train berakhir, dikeluarkan karena jendela D1-D10 nya tidak lengkap.

In [48]:
D1_MIN_SHARE = 0.5

cinemas_per_day = train.groupby(["movie_title", "date_show"])["cinema_ids"].nunique()
first_show = train.groupby("movie_title")["date_show"].min()
wide_release = cinemas_per_day[cinemas_per_day >= D1_MIN_SHARE * cinemas_per_day.groupby(level=0).transform("max")]
train_d1 = wide_release.reset_index().groupby("movie_title")["date_show"].min()
train_d1 = train_d1[
    (first_show[train_d1.index] > train["date_show"].min())
    & (train_d1 + pd.Timedelta(days=9) <= train["date_show"].max())
]
test_d1 = test_hist.groupby("movie_title")["date_show"].min()

print(f"film train: {len(train_d1)} | film test_history: {len(test_d1)} | film test: {test['movie_title'].nunique()}")

film train: 205 | film test_history: 163 | film test: 160


### Pasangan film dan bioskop
Membangun jendela D1-D10 untuk setiap pasangan dengan tiket dan jumlah show per hari. D1-D3 menjadi input dan D4-D10 menjadi target; pasangan test hanya memiliki D1-D3. Pasangan train tanpa penjualan pada D3 dibuang, sama seperti definisi pasangan test.

In [49]:
DAILY_COLS = {"total_ticket": "t", "total_show": "show"}

def build_window(hist, d1_map, n_days):
    df = hist[hist["movie_title"].isin(d1_map.index)].copy()
    df["day"] = (df["date_show"] - df["movie_title"].map(d1_map)).dt.days + 1
    df = df[df["day"].between(1, n_days)]
    parts = []
    for col, prefix in DAILY_COLS.items():
        p = df.pivot_table(index=["movie_title", "cinema_ids"], columns="day", values=col, fill_value=0)
        p = p.reindex(columns=range(1, n_days + 1), fill_value=0)
        p.columns = [f"{prefix}{d}" for d in p.columns]
        parts.append(p)
    out = pd.concat(parts, axis=1).reset_index()
    out["d1_date"] = out["movie_title"].map(d1_map)
    return out

train_window = build_window(train, train_d1, 10)
test_window = build_window(test_hist, test_d1, 3)

train_pairs = train_window[train_window["t3"] > 0].reset_index(drop=True)
test_pairs = test_window.merge(test[["movie_title", "cinema_ids"]].drop_duplicates(), on=["movie_title", "cinema_ids"])
print(f"pasangan train: {len(train_pairs)} | pasangan test: {len(test_pairs)}")

pasangan train: 8007 | pasangan test: 10373


### Format harian
Satu baris per pasangan dan hari target (horizon 4 sampai 10), sehingga satu model memprediksi semua horizon dan horizon menjadi fitur.

In [50]:
HORIZONS = range(4, 11)
INPUT_COLS = [f"{p}{d}" for p in DAILY_COLS.values() for d in (1, 2, 3)]

def to_long(pairs, with_target):
    rows = []
    for h in HORIZONS:
        r = pairs[["movie_title", "cinema_ids", "d1_date"] + INPUT_COLS].copy()
        r["horizon"] = h
        r["date_show"] = r["d1_date"] + pd.Timedelta(days=h - 1)
        r["total_ticket"] = pairs[f"t{h}"].values if with_target else np.nan
        rows.append(r)
    return pd.concat(rows, ignore_index=True)

train_long = to_long(train_pairs, with_target=True).assign(is_train=1)
test_long = to_long(test_pairs, with_target=False).assign(is_train=0).merge(
    test[["id", "movie_title", "cinema_ids", "date_show"]], on=["movie_title", "cinema_ids", "date_show"], how="left"
)
full = pd.concat([train_long, test_long], ignore_index=True)
print(f"baris train: {len(train_long)} | baris test: {len(test_long)} | baris test.csv: {len(test)}")

baris train: 56049 | baris test: 72611 | baris test.csv: 72611


### Penggabungan sumber data
Menambahkan kota tiap bioskop, tipe hari libur, batas harga tiket, dan metadata film. Akhiran 2D, 3D, dan IMAX dihapus dari judul sebelum penggabungan metadata, karena semua versi format dari satu film memiliki informasi film yang sama.

In [51]:
FORMAT_PATTERN = r"\s*\((?:IMAX )?[23]D\)\s*$"
CINEMA_CITY = pd.concat([train, test_hist]).drop_duplicates("cinema_ids").set_index("cinema_ids")["city_name"]
PRICE_DAY = {"weekday": "Weekday", "friday": "Friday", "weekend": "Weekend"}

def join_sources(df):
    n = len(df)
    df = df.copy()
    df["city_name"] = df["cinema_ids"].map(CINEMA_CITY)
    df = df.merge(holidays.rename(columns={"date": "date_show"}), on="date_show", how="left")
    df["price_day"] = df["day_tipe"].map(PRICE_DAY)
    df = df.merge(prices.rename(columns={"ceil": "price"}), on=["city_name", "price_day"], how="left")
    df["base_title"] = df["movie_title"].str.replace(FORMAT_PATTERN, "", regex=True)
    df = df.merge(movies.rename(columns={"original_title": "base_title"}), on="base_title", how="left")
    assert len(df) == n
    return df

full = join_sources(full)
print(f"full: {full.shape}")
full.isna().sum()[lambda s: s > 0]

full: (128660, 27)


total_ticket     72611
id               56049
holiday_name    122732
dtype: int64

### Penggabungan external datasets
Menambahkan cuti bersama dan libur sekolah ke setiap baris berdasarkan tanggal.

In [52]:
def join_external(df):
    n = len(df)
    df = df.merge(cuti[["date", "name"]].rename(columns={"date": "date_show", "name": "cuti_bersama_name"}), on="date_show", how="left")
    df = df.merge(school[["date", "period"]].rename(columns={"date": "date_show", "period": "school_break"}), on="date_show", how="left")
    assert len(df) == n
    return df

full = join_external(full)
print(f"full: {full.shape}")
full.groupby("is_train")[["cuti_bersama_name", "school_break"]].count()

full: (128660, 29)


,cuti_bersama_name,school_break
is_train,,
0,2392,12540
1,1141,6208


In [53]:
full.head()

,movie_title,cinema_ids,d1_date,t1,t2,t3,show1,show2,show3,horizon,...,price,base_title,age_rating,genre,producer,director,writer,casts,cuti_bersama_name,school_break
0,28 YEARS LATER,015555fbcd8faef6b55a854605df0a87,2025-06-18,261.0,210.0,137.0,18.0,17.0,12.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
1,28 YEARS LATER,0191582d75f94a412c0fa12d36bcee3f,2025-06-18,1376.0,1033.0,946.0,23.0,23.0,23.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
2,28 YEARS LATER,04a2de432a8901363a05137a8e8de1d2,2025-06-18,0.0,83.0,85.0,0.0,3.0,3.0,4,...,40000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
3,28 YEARS LATER,073b764928864ed8fa0b6b115bfdd685,2025-06-18,159.0,55.0,43.0,9.0,3.0,2.0,4,...,45000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN
4,28 YEARS LATER,108f9a4d181f12fd1b7e3c9adae38478,2025-06-18,643.0,458.0,447.0,30.0,19.0,20.0,4,...,55000,28 YEARS LATER,Dewasa,Thriller,"Danny Boyle, Alex Garland, Peter Rice",Danny Boyle,"Danny Boyle, Alex Garland","Jodie Comer, Aaron Taylor-Johnson, Jack O'Conn...",NaN,NaN


# Preprocessing

### Hapus hari yang tidak lengkap
Hari ketika train.csv memiliki baris jauh lebih sedikit dari hari-hari di sekitarnya adalah rekaman parsial. Baris train yang hari targetnya atau jendela D1-D3 nya menyentuh hari seperti itu dihapus, karena tiketnya tercatat terlalu rendah.

In [54]:
rows_per_day = train.groupby("date_show").size().reindex(
    pd.date_range(train["date_show"].min(), train["date_show"].max()), fill_value=0
)
row_share = rows_per_day / rows_per_day.rolling(15, center=True, min_periods=5).median()
GAP_DAYS = row_share.index[row_share < 0.7]

def drop_gap_rows(df):
    d13_on_gap = np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(GAP_DAYS) for k in range(3)], axis=0)
    broken = (df["is_train"] == 1) & (df["date_show"].isin(GAP_DAYS) | d13_on_gap)
    return df[~broken].reset_index(drop=True)

print("gap days:", [d.date().isoformat() for d in GAP_DAYS])
n_before = len(full)
full = drop_gap_rows(full)
print(f"dropped train rows: {n_before - len(full)} | full: {full.shape}")

gap days: ['2025-06-07', '2025-06-09', '2025-06-11', '2025-06-13', '2025-06-14', '2025-06-15', '2025-06-16']
dropped train rows: 1378 | full: (127282, 29)


### Flag kalender
Hari libur membuat lebih banyak orang datang ke bioskop. File holidays hanya menandai libur nasional, sedangkan cuti bersama dan libur sekolah berasal dari kalender eksternal, sehingga masing-masing diberi flag sendiri.

In [55]:
def add_calendar_flags(df):
    df = df.copy()
    df["is_holiday"] = (df["holiday_tipe"] == "holiday").astype(int)
    df["is_cuti_bersama"] = df["cuti_bersama_name"].notna().astype(int)
    df["is_school_holiday"] = df["school_break"].notna().astype(int)
    return df

full = add_calendar_flags(full)
full.groupby("is_train")[["is_holiday", "is_cuti_bersama", "is_school_holiday"]].mean().round(3)

,is_holiday,is_cuti_bersama,is_school_holiday
is_train,,,
0,0.041,0.033,0.173
1,0.054,0.018,0.114


### Kolom genre
Genre mengubah seberapa cepat penjualan film turun: film keluarga dan animasi bertahan lebih lama, sedangkan film horor cepat turun. Setiap genre menjadi satu kolom 0 atau 1, dan hanya genre yang muncul pada setidaknya 2% baris train yang dipakai, karena genre yang lebih jarang terlalu sedikit barisnya untuk dipelajari model.

In [56]:
def add_genres(df, genre_cols=None):
    dummies = df["genre"].str.get_dummies(sep=", ").add_prefix("genre_")
    dummies.columns = dummies.columns.str.replace(" ", "_")
    if genre_cols is not None:
        dummies = dummies.reindex(columns=genre_cols, fill_value=0)
    return pd.concat([df, dummies], axis=1), list(dummies.columns)

full, GENRE_COLS = add_genres(full)
COMMON_GENRES = [g for g in GENRE_COLS if full.loc[full["is_train"] == 1, g].mean() >= 0.02]
print(f"genres: {len(GENRE_COLS)} | used as features: {len(COMMON_GENRES)}")
full.loc[full["is_train"] == 1, GENRE_COLS].mean().sort_values(ascending=False).round(3).head(10)

genres: 27 | used as features: 15


genre_Horror       0.375
genre_Drama        0.328
genre_Action       0.212
genre_Thriller     0.146
genre_Comedy       0.120
genre_Animation    0.070
genre_Mystery      0.063
genre_Fantasy      0.056
genre_Superhero    0.052
genre_Family       0.043
dtype: float64

### Kolom kategori
city_name, day_tipe, dan age_rating dideklarasikan sebagai kategori dengan level yang sama pada train dan test, sehingga tree memperlakukannya sebagai kelompok dan bukan angka berurutan, dan kodenya sama antara train dan test.

In [57]:
CAT_COLS = ["city_name", "day_tipe", "age_rating"]
CAT_LEVELS = {col: full[col].astype("category").cat.categories for col in CAT_COLS}

def set_categories(df):
    df = df.copy()
    for col in CAT_COLS:
        df[col] = pd.Categorical(df[col], categories=CAT_LEVELS[col])
    return df

full = set_categories(full)
full[CAT_COLS].nunique()

city_name     69
day_tipe       3
age_rating     4
dtype: int64

### Hapus kolom tanpa sinyal
Kolom kredit (producer, director, writer, casts) hampir unik untuk setiap film sehingga model hanya bisa menghafalnya. Kolom lain yang dihapus hanya dipakai untuk penggabungan atau mengulang informasi yang sudah ter-encode.

In [58]:
DROP_COLS = [
    "holiday_tipe", "holiday_name", "cuti_bersama_name", "school_break", "genre",
    "price_day", "base_title", "producer", "director", "writer", "casts",
]

def drop_unused(df):
    return df.drop(columns=DROP_COLS)

full = drop_unused(full)
print(f"full: {full.shape}")
print(list(full.columns))

full: (127282, 48)
['movie_title', 'cinema_ids', 'd1_date', 't1', 't2', 't3', 'show1', 'show2', 'show3', 'horizon', 'date_show', 'total_ticket', 'is_train', 'id', 'city_name', 'day_tipe', 'price', 'age_rating', 'is_holiday', 'is_cuti_bersama', 'is_school_holiday', 'genre_Action', 'genre_Adventure', 'genre_Animation', 'genre_Biography', 'genre_Comedy', 'genre_Crime', 'genre_Disaster', 'genre_Documentary', 'genre_Drama', 'genre_Family', 'genre_Fantasy', 'genre_Gothic', 'genre_History', 'genre_Horror', 'genre_Music', 'genre_Musical', 'genre_Mystery', 'genre_Religi', 'genre_Romance', 'genre_Sci-Fi', 'genre_Sport', 'genre_Superhero', 'genre_Survival', 'genre_Suspense', 'genre_Teen', 'genre_Thriller', 'genre_War']


### Target
target_ratio adalah tiket pada hari target dibagi scale, yaitu rata-rata D1-D3 (minimal 1). Metrik kompetisi adalah selisih absolut dibagi scale yang sama, sehingga mean absolute error dari target_ratio adalah metriknya sendiri, dan pasangan dengan ukuran yang sangat berbeda menjadi sebanding.

In [59]:
def add_target(df):
    df = df.copy()
    df["scale"] = df[["t1", "t2", "t3"]].mean(axis=1).clip(lower=1)
    df["target_ratio"] = df["total_ticket"] / df["scale"]
    return df

full = add_target(full)
full.loc[full["is_train"] == 1].groupby("horizon")["target_ratio"].median().round(3)

horizon
4     0.915
5     0.616
6     0.467
7     0.360
8     0.163
9     0.000
10    0.000
Name: target_ratio, dtype: float64

# Feature Engineering

### Blok hari libur
Penonton tetap banyak selama rangkaian hari libur berturut-turut dan turun ketika rangkaian itu berakhir, sehingga long weekend berperilaku berbeda dari akhir pekan biasa.
- off_block_len: panjang rangkaian hari libur tanpa putus (akhir pekan, libur nasional, cuti bersama) yang memuat hari target, bernilai 0 untuk hari kerja.

In [60]:
cal = pd.DataFrame({"date": pd.date_range(full["d1_date"].min() - pd.Timedelta(days=30), full["date_show"].max())})
NATIONAL_HOLIDAYS = pd.concat([holidays.loc[holidays["holiday_tipe"] == "holiday", "date"], extra_holidays["date"]])
cal["off"] = (
    (cal["date"].dt.dayofweek >= 5) | cal["date"].isin(NATIONAL_HOLIDAYS) | cal["date"].isin(cuti["date"])
).astype(int)
block_id = (cal["off"] != cal["off"].shift()).cumsum()
cal["block_len"] = cal.groupby(block_id)["off"].transform("size") * cal["off"]
BLOCK_LEN = cal.set_index("date")["block_len"]

def add_off_block(df):
    return df.assign(off_block_len=df["date_show"].map(BLOCK_LEN))

full = add_off_block(full)

print("longest blocks:")
print(cal[cal["block_len"] >= 4].groupby(block_id)["date"].agg(["min", "max", "size"]).reset_index(drop=True))
full.groupby("is_train")["off_block_len"].value_counts(normalize=True).unstack().round(3)

longest blocks:
         min        max  size
0 2025-03-28 2025-04-07    11
1 2025-05-10 2025-05-13     4
2 2025-05-29 2025-06-01     4
3 2025-06-06 2025-06-09     4
4 2025-12-25 2025-12-28     4
5 2026-02-14 2026-02-17     4
6 2026-03-18 2026-03-24     7


off_block_len,0,1,2,3,4,7,11
is_train,,,,,,,
0,0.658,0.005,0.228,0.019,0.047,0.042,NaN
1,0.657,0.006,0.213,0.072,0.051,NaN,0.001


### Fitur pasangan
Perilaku pasangan itu sendiri pada D1-D3, informasi terkuat untuk hari-hari berikutnya:
- trend: tiket D3 relatif terhadap D1, apakah permintaan pasangan naik atau turun.
- t3_share: porsi tiket D1-D3 yang terjual pada D3, menandakan puncak yang terlambat atau mulai tayang terlambat.
- show_mean: rata-rata jumlah show per hari, kapasitas yang diberikan bioskop untuk film tersebut.
- tickets_per_show3: tiket per show pada D3, permintaan yang tidak bergantung pada jumlah show yang dijadwalkan.
- active_days: jumlah hari dengan penjualan pada D1-D3, pasangan yang mulai terlambat memiliki lebih sedikit.

In [61]:
D13 = ["t1", "t2", "t3"]

def add_pair_features(df):
    df = df.copy()
    df["trend"] = (df["t3"] + 1) / (df["t1"] + 1)
    df["t3_share"] = df["t3"] / df[D13].sum(axis=1)
    df["show_mean"] = df[["show1", "show2", "show3"]].mean(axis=1)
    df["tickets_per_show3"] = df["t3"] / df["show3"].replace(0, np.nan)
    df["active_days"] = (df[D13] > 0).sum(axis=1)
    return df

full = add_pair_features(full)
full[["trend", "t3_share", "show_mean", "tickets_per_show3", "active_days"]].describe().round(2)

,trend,t3_share,show_mean,tickets_per_show3,active_days
count,127282.00,127282.00,127282.00,127282.00,127282.00
mean,11.13,0.33,8.34,32.42,2.90
std,51.19,0.16,11.10,31.17,0.36
min,0.01,0.00,0.33,0.25,1.00
25%,0.56,0.24,3.33,12.14,3.00
50%,0.89,0.32,5.00,24.00,3.00
75%,1.49,0.40,9.00,42.20,3.00
max,1199.00,1.00,216.00,421.33,3.00


### Fitur film
Film yang sama di semua bioskop, dijumlahkan dari D1-D3. Film diprediksi lebih baik dari gambaran nasionalnya daripada dari satu bioskop saja.
- mv_t1, mv_t2, mv_t3: tiket nasional film pada masing-masing D1-D3, kekuatan film.
- mv_clusters: jumlah bioskop yang memiliki penjualan pada D3, luasnya rilis.
- mv_trend: D3 nasional relatif terhadap D1, tren yang lebih stabil daripada tren satu pasangan.
- pair_share: porsi tiket D1-D3 nasional film yang dijual oleh bioskop ini, seberapa penting bioskop tersebut bagi film.

In [62]:
def film_features(window):
    g = window.groupby("movie_title")
    out = g[D13].sum().add_prefix("mv_")
    out["mv_clusters"] = g["t3"].apply(lambda s: (s > 0).sum())
    out["mv_trend"] = (out["mv_t3"] + 1) / (out["mv_t1"] + 1)
    return out

FILM_COLS = ["mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend"]

def add_film_features(df, window):
    df = df.copy()
    feats = film_features(window)
    for col in FILM_COLS:
        df[col] = df["movie_title"].map(feats[col]).values
    df["pair_share"] = df[D13].sum(axis=1) / df[["mv_t1", "mv_t2", "mv_t3"]].sum(axis=1)
    return df

is_tr = full["is_train"] == 1
full = pd.concat([add_film_features(full[is_tr], train_window), add_film_features(full[~is_tr], test_window)]).sort_index()
full.groupby("is_train")[FILM_COLS + ["pair_share"]].median().round(3)

,mv_t1,mv_t2,mv_t3,mv_clusters,mv_trend,pair_share
is_train,,,,,,
0,11615.0,12300.0,12249.0,87.0,0.869,0.008
1,22042.0,18511.0,17704.0,81.0,0.923,0.010


### Fitur bioskop
Ukuran dan program tayang yang biasa dari bioskop, dihitung dari jendela D1-D3 semua film tanpa informasi target:
- cl_scale: rata-rata scale D1-D3 film yang tayang, bioskop besar menurun berbeda dari bioskop kecil.
- cl_movies: jumlah film berbeda yang tayang, berapa banyak film yang bersaing untuk layar yang sama.

In [63]:
all_windows = pd.concat([train_window, test_window], ignore_index=True)
cinema_feats = (
    all_windows.assign(pair_scale=all_windows[D13].mean(axis=1))
    .groupby("cinema_ids")
    .agg(cl_scale=("pair_scale", "mean"), cl_movies=("movie_title", "nunique"))
)

def add_cinema_features(df):
    return df.join(cinema_feats, on="cinema_ids")

full = add_cinema_features(full)
cinema_feats.describe().round(2)

,cl_scale,cl_movies
count,121.00,121.00
mean,192.95,192.13
std,165.41,73.64
min,18.77,5.00
25%,106.63,145.00
50%,160.53,189.00
75%,221.98,239.00
max,1348.26,347.00


### Fitur hari dan harga
- dow: hari dalam seminggu dari hari target, akhir pekan lebih ramai.
- d1_dow: hari dalam seminggu dari D1, posisi akhir pekan di dalam D1-D10 mengubah bentuk penurunan.
- price_ratio: harga tiket pada hari target relatif terhadap rata-rata harga D1-D3, hari yang lebih murah menarik lebih banyak orang.

In [64]:
DAY_TIPE_BY_DATE = holidays.set_index("date")["day_tipe"]
PRICE_MAP = prices.set_index(["city_name", "price_day"])["ceil"]

def add_weekday_price(df):
    df = df.copy()
    df["dow"] = df["date_show"].dt.dayofweek
    df["d1_dow"] = df["d1_date"].dt.dayofweek

    def price_on(dates):
        keys = pd.MultiIndex.from_arrays([df["city_name"].astype(str), dates.map(DAY_TIPE_BY_DATE).map(PRICE_DAY)])
        return pd.Series(PRICE_MAP.reindex(keys).values, index=df.index)

    d13_price = pd.concat([price_on(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=1).mean(axis=1)
    df["price_ratio"] = df["price"] / d13_price
    return df

full = add_weekday_price(full)
full[["dow", "d1_dow", "price_ratio"]].describe().round(3)

,dow,d1_dow,price_ratio
count,127282.0,127282.000,127282.000
mean,3.0,2.772,0.977
std,2.0,0.726,0.144
min,0.0,0.000,0.615
25%,1.0,2.000,0.875
50%,3.0,3.000,0.955
75%,5.0,3.000,1.105
max,6.0,6.000,1.500


### Hari libur di sekitar jendela D1-D3 dan hari target
Penjualan D1-D3 terdongkrak bila jatuh pada hari libur, dan hari target bisa berada pada situasi kalender yang berbeda, sehingga model perlu tahu keduanya:
- off_d1, off_d2, off_d3: apakah masing-masing hari pada jendela adalah hari libur.
- win_off, win_hol, win_wk: jumlah hari libur, libur nasional, dan hari akhir pekan pada jendela.
- tgt_off: apakah hari target adalah hari libur.
- off_rel: hari target libur dikurangi porsi hari libur pada jendela, jendela yang penuh hari libur lalu diikuti hari kerja memprediksi penurunan.
- tgt_prev_off, tgt_next_off: apakah hari di samping hari target libur, tepi dari long weekend.
- between_off, between_off_share: jumlah dan porsi hari libur antara D4 dan hari target.

In [65]:
def is_off(dates):
    return ((dates.dt.dayofweek >= 5) | dates.isin(NATIONAL_HOLIDAYS) | dates.isin(cuti["date"])).astype(int)

def add_window_offdays(df):
    """days off (weekend, national holiday, cuti bersama) in the D1-D3 window, on and around the target day, and between D4 and the target"""
    df = df.copy()
    day = lambda k: df["d1_date"] + pd.Timedelta(days=k)
    for k in range(3):
        df[f"off_d{k + 1}"] = is_off(day(k))
    df["win_off"] = df[["off_d1", "off_d2", "off_d3"]].sum(axis=1)
    df["win_hol"] = sum(day(k).isin(NATIONAL_HOLIDAYS).astype(int) for k in range(3))
    df["win_wk"] = sum((day(k).dt.dayofweek >= 5).astype(int) for k in range(3))
    df["tgt_off"] = is_off(df["date_show"])
    df["off_rel"] = df["tgt_off"] - df["win_off"] / 3
    df["tgt_prev_off"] = is_off(df["date_show"] - pd.Timedelta(days=1))
    df["tgt_next_off"] = is_off(df["date_show"] + pd.Timedelta(days=1))
    df["between_off"] = sum(((df["horizon"] > h) & (is_off(day(h - 1)) == 1)).astype(int) for h in range(4, 11))
    df["between_off_share"] = df["between_off"] / (df["horizon"] - 4).clip(lower=1)
    return df

OFF_COLS = ["off_d1", "off_d2", "off_d3", "win_off", "win_hol", "win_wk", "tgt_off", "off_rel",
            "tgt_prev_off", "tgt_next_off", "between_off", "between_off_share"]
full = add_window_offdays(full)
full.groupby("is_train")[OFF_COLS].mean().round(3)

,off_d1,off_d2,off_d3,win_off,win_hol,win_wk,tgt_off,off_rel,tgt_prev_off,tgt_next_off,between_off,between_off_share
is_train,,,,,,,,,,,,
0,0.078,0.290,0.692,1.059,0.178,0.723,0.342,-0.011,0.353,0.342,1.200,0.425
1,0.032,0.318,0.749,1.098,0.241,0.838,0.343,-0.023,0.345,0.340,1.118,0.404


### Profil film
Jenis film dari external/film_profile.csv, karena penonton film horor lokal, film asing blockbuster, dan sekuel bertahan berbeda:
- is_local: film Indonesia atau asing.
- is_sequel: bagian dari sekuel atau franchise, yang membawa penonton yang sudah ada.
- kind_code: live action, animasi, anime, rilis ulang, konser, atau dokumenter.
- source_code: cerita orisinal, novel, komik, remake, kisah nyata, atau sumber lain.
- origin_code: negara asal.
- predecessor_adm: jumlah penonton film sebelumnya dalam franchise, ukuran penonton yang sudah ada.
- local_horror: lokal dan horor sekaligus, genre yang dibuka kuat dan cepat turun.
Fakta yang baru dipublikasikan setelah 2025-09-30 dikosongkan, sehingga tidak ada informasi setelah cutoff yang masuk ke model.

In [66]:
PROFILE_CUTOFF = pd.Timestamp("2025-09-30")
FILM_KINDS = ["live_action", "animation", "anime", "rerelease", "concert", "documentary"]
SOURCES = ["original", "novel", "other", "comic", "remake", "true_story", "viral_story", "folklore", "game"]
ORIGINS = ["Indonesia", "United States", "South Korea", "Japan", "Thailand"]

def load_film_profile():
    p = pd.read_csv(f"{EXTERNAL_DIR}/film_profile.csv")
    late = (p["split"] == "test") & ((p["info_public_before_cutoff"] == "no") | (pd.to_datetime(p["fact_source_date"], format="mixed", errors="coerce") > PROFILE_CUTOFF))
    pred_known = pd.to_datetime(p["predecessor_published"], errors="coerce") <= PROFILE_CUTOFF
    first_origin = p["origin_country"].astype(str).str.split(",").str[0].str.strip()
    prof = pd.DataFrame({
        "movie_title": p["movie_title"],
        "is_local": p["is_local"].astype(float),
        "is_sequel": p["is_sequel_or_franchise"].astype(float),
        "kind_code": p["film_kind"].map({k: i for i, k in enumerate(FILM_KINDS)}).astype(float),
        "source_code": p["source_material"].map({k: i for i, k in enumerate(SOURCES)}).astype(float),
        "origin_code": first_origin.map({k: i for i, k in enumerate(ORIGINS)}).fillna(len(ORIGINS)).astype(float),
        "predecessor_adm": np.where(pred_known, np.log1p(p["predecessor_admissions_id"]), np.nan),
    })
    prof.loc[late.values, prof.columns[1:]] = np.nan
    return prof.drop_duplicates("movie_title").set_index("movie_title")

FILM_PROFILE = load_film_profile()
PROFILE_COLS = list(FILM_PROFILE.columns) + ["local_horror"]

def add_film_profile(df):
    df = df.copy()
    for c in FILM_PROFILE.columns:
        df[c] = df["movie_title"].map(FILM_PROFILE[c])
    df["local_horror"] = df["is_local"] * df["genre_Horror"]
    return df

full = add_film_profile(full)
print("films without a profile:", full.loc[full["is_local"].isna()].groupby("is_train")["movie_title"].nunique().to_dict())
full.groupby("is_train")[PROFILE_COLS].mean().round(3)

films without a profile: {0: 8}


,is_local,is_sequel,kind_code,source_code,origin_code,predecessor_adm,local_horror
is_train,,,,,,,
0,0.529,0.337,0.099,1.197,1.229,14.955,0.206
1,0.564,0.309,0.135,2.052,0.753,14.545,0.286


### Pasangan yang mulai terlambat
Sebuah bioskop bisa menambahkan film setelah D1, sehingga pasangan dengan penjualan hanya pada D3 lebih muda daripada yang ditunjukkan horizonnya.
- first_active: hari pertama pada D1-D3 yang memiliki penjualan.
- eff_age: horizon dikurangi first_active ditambah 1, usia sebenarnya dari pasangan pada hari target.
- last_ratio: penjualan pada hari aktif terakhir relatif terhadap scale, level tempat pasangan mengakhiri D1-D3.
- active_mean: rata-rata penjualan hanya pada hari aktif, karena rata-rata D1-D3 meremehkan pasangan yang mulai terlambat.

In [67]:
def add_late_start(df):
    df = df.copy()
    t = df[D13].values
    first = np.select([t[:, 0] > 0, t[:, 1] > 0, t[:, 2] > 0], [1, 2, 3], 4)
    last = np.select([t[:, 2] > 0, t[:, 1] > 0], [t[:, 2], t[:, 1]], t[:, 0])
    df["first_active"] = first
    df["eff_age"] = df["horizon"] - first + 1
    df["last_ratio"] = last / df["scale"]
    df["active_mean"] = t.sum(axis=1) / np.clip((t > 0).sum(axis=1), 1, None) / df["scale"]
    return df

full = add_late_start(full)
full.groupby("is_train")["first_active"].value_counts(normalize=True).unstack().round(3)

first_active,1,2,3
is_train,,,
0,0.916,0.066,0.018
1,0.928,0.055,0.018


### Persaingan dari film lain yang baru rilis
Sebuah film kehilangan layar dan penonton ke film yang rilis setelahnya, dan hari pasar yang kuat mengangkat semua film. Fitur ini hanya memakai informasi D1-D3 film lain, tidak pernah targetnya.
- comp_size_cin: ukuran film yang rilis di bioskop yang sama setelah D1 film ini dan sampai hari target, relatif terhadap scale film itu sendiri.
- comp_fresh_cin: sama, tetapi hanya untuk film dalam tiga hari pertamanya, persaingan layar yang paling kuat.
- comp_size_nat: hitungan yang sama pada tingkat nasional.
- date_idx: seberapa kuat pembukaan film lain pada tanggal target dibandingkan bentuk pembukaan yang biasa untuk hari rilisnya, ukuran seberapa ramai pasar hari itu.
- own_idx: ukuran yang sama pada hari D1-D3 film itu sendiri.
- date_idx_rel: date_idx dikurangi own_idx, apakah hari target lebih ramai atau lebih sepi dari hari-hari saat film diamati.

In [68]:
COMP_MIN_SIZE = 50

def opening_pairs(df):
    return df.drop_duplicates(["movie_title", "cinema_ids"])[["movie_title", "cinema_ids", "d1_date", "t1", "t2", "t3", "scale"]]

def film_openings(df):
    f = opening_pairs(df).groupby(["movie_title", "d1_date"])[D13].sum().reset_index()
    f["m"] = f[D13].mean(axis=1)
    return f[f["m"] >= COMP_MIN_SIZE].reset_index(drop=True)

def typical_opening_shape(df):
    """usual film D_k / mean(D1-D3), k = 1..3, by release weekday"""
    f = film_openings(df)
    log_r = lambda g, k: np.average(np.log((g[f"t{k}"] + 1) / (g["m"] + 1)), weights=np.sqrt(g["m"]))
    overall = [np.exp(log_r(f, k)) for k in (1, 2, 3)]
    by_dow = {dw: [np.exp(log_r(g, k)) for k in (1, 2, 3)] for dw, g in f.groupby(f["d1_date"].dt.dayofweek) if len(g) >= 5}
    return {dw: by_dow.get(dw, overall) for dw in range(7)}

def opening_surprise(df, shape):
    """log of each film's D1-D3 days vs the usual shape for its release weekday, one row per film and date"""
    rows = []
    for r in film_openings(df).itertuples():
        t = shape[r.d1_date.dayofweek]
        for k in range(3):
            rows.append((r.movie_title, r.d1_date + pd.Timedelta(days=k), np.log((getattr(r, f"t{k + 1}") + 1) / (r.m + 1) / t[k]), np.sqrt(r.m)))
    s = pd.DataFrame(rows, columns=["movie_title", "date", "s", "w"])
    return s.assign(sw=s["s"] * s["w"])

def date_strength(dates, titles, surprise):
    """size-weighted mean surprise of the other films in their D1-D3 on each date (NaN when there are none)"""
    tot = surprise.groupby("date")[["sw", "w"]].sum().reindex(pd.DatetimeIndex(dates)).values
    own = surprise.set_index(["date", "movie_title"])[["sw", "w"]].reindex(pd.MultiIndex.from_arrays([pd.DatetimeIndex(dates), titles])).fillna(0).values
    w = tot[:, 1] - own[:, 1]
    ok = w > 1e-9
    return np.where(ok, (tot[:, 0] - own[:, 0]) / np.where(ok, w, 1), np.nan)

def add_competition(df, ref, shape):
    """ref: rows whose pairs are the other films' openings"""
    df = df.reset_index(drop=True)
    others = opening_pairs(ref)
    own_size = opening_pairs(df).groupby("movie_title")["scale"].sum()

    key = df[["movie_title", "cinema_ids", "d1_date", "date_show"]].reset_index()
    m = key.merge(others[["cinema_ids", "movie_title", "d1_date", "scale"]], on="cinema_ids", suffixes=("", "_o"))
    m = m[(m["movie_title_o"] != m["movie_title"]) & (m["d1_date_o"] > m["d1_date"]) & (m["d1_date_o"] <= m["date_show"])]
    fresh = m[(m["date_show"] - m["d1_date_o"]).dt.days <= 2].groupby("index")["scale"].sum()
    df["comp_size_cin"] = np.log1p(m.groupby("index")["scale"].sum().reindex(df.index, fill_value=0).values / df["scale"].values)
    df["comp_fresh_cin"] = np.log1p(fresh.reindex(df.index, fill_value=0).values / df["scale"].values)

    films = others.groupby("movie_title").agg(fd1=("d1_date", "min"), fsize=("scale", "sum")).rename_axis("movie_o").reset_index()
    u = df[["movie_title", "d1_date", "date_show"]].drop_duplicates()
    mm = u.merge(films, how="cross")
    mm = mm[(mm["movie_o"] != mm["movie_title"]) & (mm["fd1"] > mm["d1_date"]) & (mm["fd1"] <= mm["date_show"])]
    u = u.merge(mm.groupby(["movie_title", "d1_date", "date_show"]).agg(comp_size_nat=("fsize", "sum")).reset_index(),
                how="left", on=["movie_title", "d1_date", "date_show"])

    surprise = opening_surprise(ref, shape)
    u["date_idx"] = date_strength(u["date_show"], u["movie_title"].values, surprise)
    own_days = np.vstack([date_strength(u["d1_date"] + pd.Timedelta(days=k), u["movie_title"].values, surprise) for k in range(3)])
    n_days = (~np.isnan(own_days)).sum(axis=0)
    u["own_idx"] = np.where(n_days > 0, np.nansum(own_days, axis=0) / np.maximum(n_days, 1), np.nan)
    u["date_idx_rel"] = u["date_idx"] - u["own_idx"]

    df = df.merge(u, how="left", on=["movie_title", "d1_date", "date_show"])
    df["comp_size_nat"] = np.log1p(df["comp_size_nat"].fillna(0) / df["movie_title"].map(own_size))
    return df

COMP_COLS = ["comp_size_cin", "comp_fresh_cin", "comp_size_nat", "date_idx", "own_idx", "date_idx_rel"]
OPENING_SHAPE = typical_opening_shape(full[full["is_train"] == 1])
is_tr = full["is_train"] == 1
full = pd.concat([add_competition(full[is_tr], full[is_tr], OPENING_SHAPE),
                  add_competition(full[~is_tr], full[~is_tr], OPENING_SHAPE)], ignore_index=True)
full.groupby("is_train")[COMP_COLS].mean().round(3)

,comp_size_cin,comp_fresh_cin,comp_size_nat,date_idx,own_idx,date_idx_rel
is_train,,,,,,
0,1.301,0.898,1.438,0.005,-0.012,0.020
1,1.256,0.874,1.485,-0.000,0.004,-0.003


### Baris latih tambahan (jendela bergeser)
Hanya ada sekitar dua ratus pembukaan film untuk dipelajari. Menggeser D1 lebih lambat 1 sampai 14 hari memperlakukan hari yang lebih akhir dari film yang sama sebagai D1 dan menghasilkan contoh latih baru dengan tiket nyata, mencakup tahap lain dari masa tayang film dan hari kalender lain.
- age_offset: jumlah hari D1 digeser, 0 untuk baris asli dan baris test, sehingga model dapat membedakan baris hasil geser.
Baris geser hanya dipakai untuk fitting, tidak pernah untuk validasi, dan bobotnya lebih rendah dari baris asli pada model tree karena kurang khas dibanding pembukaan yang sebenarnya.

In [69]:
AUG_SHIFTS = [1, 2, 3, 5, 7, 10, 14]

def build_shifted_rows(shift):
    d1 = train_d1 + pd.Timedelta(days=shift)
    d1 = d1[d1 + pd.Timedelta(days=9) <= train["date_show"].max()]
    window = build_window(train, d1, 10)
    df = to_long(window[window["t3"] > 0].reset_index(drop=True), with_target=True).assign(is_train=1)
    df = drop_gap_rows(join_external(join_sources(df)))
    df, _ = add_genres(add_calendar_flags(df), GENRE_COLS)
    df = add_target(drop_unused(set_categories(df)))
    df = add_weekday_price(add_cinema_features(add_film_features(add_pair_features(add_off_block(df)), window)))
    df = add_film_profile(add_late_start(add_window_offdays(df)))
    df = add_competition(df, full[full["is_train"] == 1], OPENING_SHAPE)
    return df.assign(age_offset=shift)

full["age_offset"] = 0
AUG = pd.concat([build_shifted_rows(s) for s in AUG_SHIFTS], ignore_index=True)
print(f"extra rows: {len(AUG)} ({len(AUG) / (full['is_train'] == 1).sum():.1f}x the real train rows)")
AUG.groupby("age_offset")["target_ratio"].agg(["size", "mean"]).round(3)

extra rows: 254483 (4.7x the real train rows)


,size,mean
age_offset,,
1,50854,0.464
2,50833,0.482
3,48573,0.521
5,33873,0.635
7,28516,0.591
10,26786,0.410
14,15048,0.591


# Regressor Models

### Setup dan validasi
Semua model memprediksi target_ratio. Validasi memakai 5-fold yang dikelompokkan per film, sehingga film validasi tidak pernah dilihat saat training, seperti film test. Baris train yang menyentuh hari tidak lengkap dihapus, juga dari baris tambahan, karena tiketnya tercatat terlalu rendah. Prediksi di bawah ZERO_SNAP dijadikan tepat 0: sebagian besar target bernilai tepat 0 dan tree memprediksi angka positif kecil di sana.

In [70]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

RUN_MODELS = ["lgbm", "kumo", "nn"]
MODEL_SEEDS = [42, 7, 123]
ZERO_SNAP = {"lgbm": 0.2, "nn": 0.2}
N_FOLDS = 5
USE_EXTRA_ROWS = True
EXTRA_WEIGHT = 0.5

NUM_FEATURES = [
    "t1", "t2", "t3", "show1", "show2", "show3", "horizon", "scale", "price",
    "trend", "t3_share", "show_mean", "tickets_per_show3", "active_days",
    "mv_t1", "mv_t2", "mv_t3", "mv_clusters", "mv_trend", "pair_share", "cl_scale", "cl_movies",
    "dow", "d1_dow", "price_ratio",
    "is_holiday", "is_cuti_bersama", "is_school_holiday", "off_block_len",
] + OFF_COLS + [
    "first_active", "eff_age", "last_ratio", "active_mean",
] + COMP_COLS + ["age_offset"] + COMMON_GENRES + PROFILE_COLS
FEATURES = NUM_FEATURES + CAT_COLS

def incomplete_days(raw):
    """days on which train.csv is partial (found by a rule): far fewer shows or cinemas than the days around it (21-day median), or no rows at all"""
    g = raw.groupby("date_show").agg(shows=("total_show", "sum"), cinemas=("cinema_ids", "nunique")).reindex(pd.date_range(raw["date_show"].min(), raw["date_show"].max()))
    rel_shows = g["shows"] / g["shows"].rolling(21, center=True, min_periods=7).median()
    rel_cinemas = g["cinemas"] / g["cinemas"].rolling(21, center=True, min_periods=7).median()
    return pd.DatetimeIndex(g.index[(rel_shows < 0.75) | (rel_cinemas < 0.85) | g["shows"].isna()])

INCOMPLETE_DAYS = incomplete_days(train)

def touches_incomplete(df):
    """True for rows whose target day or any of D1-D3 is an incomplete day: their target and their D1-D3 base are under-reported"""
    return (df["date_show"].isin(INCOMPLETE_DAYS) | np.any([(df["d1_date"] + pd.Timedelta(days=k)).isin(INCOMPLETE_DAYS) for k in range(3)], axis=0)).values

drop_rows = (full["is_train"] == 1) & touches_incomplete(full)
print(f"incomplete train days: {len(INCOMPLETE_DAYS)} ({INCOMPLETE_DAYS.min().date()} .. {INCOMPLETE_DAYS.max().date()}); train rows dropped: {int(drop_rows.sum())} of {int((full['is_train'] == 1).sum())}")
train_df = full[(full["is_train"] == 1) & ~drop_rows].reset_index(drop=True)
test_df = full[full["is_train"] == 0].reset_index(drop=True)
CINEMA_CODE = {c: k for k, c in enumerate(sorted(pd.concat([train_df["cinema_ids"], test_df["cinema_ids"]]).unique()))}
for df_ in (train_df, test_df):
    df_["cinema_code"] = df_["cinema_ids"].map(CINEMA_CODE)
X, y, groups = train_df[FEATURES + ["cinema_code"]], train_df["target_ratio"].values, train_df["movie_title"]
X_test = test_df[FEATURES + ["cinema_code"]]
AUG = AUG[~touches_incomplete(AUG)].reset_index(drop=True)
AUG["cinema_code"] = AUG["cinema_ids"].map(CINEMA_CODE)
X_aug, y_aug = AUG[FEATURES + ["cinema_code"]], AUG["target_ratio"].values
FOLDS = list(GroupKFold(n_splits=N_FOLDS).split(X, y, groups))

OOF, TEST_PRED, SCORES = {}, {}, {}

def mase_ratio(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))

def fit_rows(idx):
    """real rows idx + the shifted-window rows of the same films; is_extra marks the shifted rows"""
    X_fit, y_fit = X.iloc[idx], y[idx]
    if not USE_EXTRA_ROWS:
        return X_fit, y_fit, np.zeros(len(idx), bool)
    keep = AUG["movie_title"].isin(set(groups.iloc[idx])).values
    return (pd.concat([X_fit, X_aug[keep]], ignore_index=True), np.r_[y_fit, y_aug[keep]],
            np.r_[np.zeros(len(idx), bool), np.ones(keep.sum(), bool)])

def extra_weights(is_extra):
    return np.where(is_extra, EXTRA_WEIGHT, 1.0)

def seed_average(make_model, X_tr, y_tr, X_pred, prep=lambda d: d, sample_weight=None):
    preds = [make_model(seed).fit(prep(X_tr), y_tr, sample_weight=sample_weight).predict(prep(X_pred)) for seed in MODEL_SEEDS]
    return np.mean(preds, axis=0)

def cross_validate(name, fit_predict):
    snap = lambda p: np.where(p < ZERO_SNAP.get(name, 0.0), 0.0, p)
    oof = np.zeros(len(train_df))
    fold_scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        oof[va_idx] = snap(np.clip(fit_predict(*fit_rows(tr_idx), X.iloc[va_idx]), 0, None))
        fold_scores.append(mase_ratio(y[va_idx], oof[va_idx]))
        print(f"{name} fold {fold}: MASE={fold_scores[-1]:.4f}")
    OOF[name], SCORES[name] = oof, fold_scores
    print(f"{name} CV MASE: {mase_ratio(y, oof):.4f}")
    TEST_PRED[name] = snap(np.clip(fit_predict(*fit_rows(np.arange(len(X))), X_test), 0, None))

print(f"features: {len(FEATURES)} | train rows: {len(train_df)} | extra rows: {len(AUG) if USE_EXTRA_ROWS else 0} | test rows: {len(test_df)}")

incomplete train days: 13 (2025-06-01 .. 2025-06-16); train rows dropped: 1766 of 54671
features: 77 | train rows: 52905 | extra rows: 244229 | test rows: 72611


### LGBM Regressor
Gradient boosted trees dengan objective absolute error yang sesuai dengan metrik. Baris tambahan diberi bobot sampel lebih rendah dari baris asli, dan prediksi beberapa seed dirata-rata agar hasil tidak bergantung pada satu seed yang kebetulan bagus.

In [71]:
LGBM_PARAMS = dict(
    n_estimators=1500,
    learning_rate=0.0149601,
    num_leaves=87,
    min_child_samples=11,
    subsample=0.641476,
    colsample_bytree=0.48309,
    reg_lambda=1.85904,
    reg_alpha=11.5708,
    subsample_freq=1,
)

def lgbm_fit_predict(X_tr, y_tr, is_extra, X_pred):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    make_model = lambda seed: LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS)
    return seed_average(make_model, X_tr, y_tr, X_pred, sample_weight=extra_weights(is_extra))

if "lgbm" in RUN_MODELS:
    cross_validate("lgbm", lgbm_fit_predict)

lgbm fold 0: MASE=0.3313
lgbm fold 1: MASE=0.3533
lgbm fold 2: MASE=0.2117
lgbm fold 3: MASE=0.3465
lgbm fold 4: MASE=0.4081
lgbm CV MASE: 0.3301


### Bobot Kumo-Tabular
Kumo-Tabular adalah model pretrained dari NVIDIA. Model ini tidak dilatih di sini dan bukan bagian dari model yang diekspor: cell ini mengunduh bobotnya dari Hugging Face, dan tidak melakukan apa pun bila sudah tersimpan di cache.

In [72]:
KUMO_REPO = "nvidia/Kumo-Tabular"
if "kumo" in RUN_MODELS:
    print("Kumo-Tabular weights at", snapshot_download(KUMO_REPO))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Kumo-Tabular weights at C:\Users\leouw\.cache\huggingface\hub\models--nvidia--Kumo-Tabular\snapshots\3520742f7f3b937a2751266cea22d7fb6686882c


### Kumo-Tabular
Model in-context pretrained: sampel baris train (baris asli dan baris geser) diberikan sebagai konteks dan tidak ada bobot yang diperbarui. Prediksinya adalah median dari quantile yang diprediksi, yang sesuai dengan metrik absolute error. Beberapa konteks acak dan beberapa pass dirata-rata karena satu sampel konteks saja menambah variansi.

In [73]:
KUMO_CONTEXT_ROWS = 9000
KUMO_EXTRA_ROWS = 3000
KUMO_CONTEXT_SEEDS = [42, 7, 123]
KUMO_ESTIMATORS = 8
KUMO_STYPES = {c: "numerical" for c in NUM_FEATURES} | {c: "categorical" for c in CAT_COLS}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
kumo_model = sdm.models.KumoTabular(task="regression", size="large", device=DEVICE).eval() if "kumo" in RUN_MODELS else None

def kumo_fit_predict(X_tr, y_tr, is_extra, X_pred, chunk_rows=2000):
    X_tr, X_pred = X_tr[FEATURES], X_pred[FEATURES]
    real_idx, extra_idx = np.flatnonzero(~is_extra), np.flatnonzero(is_extra)
    preds = []
    bar = tqdm(total=len(KUMO_CONTEXT_SEEDS) * int(np.ceil(len(X_pred) / chunk_rows)), desc="kumo", unit="chunk")
    for seed in KUMO_CONTEXT_SEEDS:
        rng = np.random.default_rng(seed)
        ctx = np.r_[rng.choice(real_idx, min(KUMO_CONTEXT_ROWS, len(real_idx)), replace=False),
                    rng.choice(extra_idx, min(KUMO_EXTRA_ROWS, len(extra_idx)), replace=False)].astype(int)
        x_ctx = TableTensor.from_pandas(X_tr.iloc[ctx], KUMO_STYPES, device=DEVICE)
        y_ctx = TableTensor.from_pandas(pd.DataFrame({"y": y_tr[ctx]}), {"y": "numerical"}, device=DEVICE)
        kumo_model.fit(x_ctx, y_ctx, num_estimators=KUMO_ESTIMATORS, generator=torch.Generator(device=DEVICE).manual_seed(seed))
        quantiles = []
        for start in range(0, len(X_pred), chunk_rows):
            x_q = TableTensor.from_pandas(X_pred.iloc[start:start + chunk_rows], KUMO_STYPES, device=DEVICE)
            quantiles.append(kumo_model.predict(x_q).numerical.float().cpu())
            bar.update(1); bar.set_postfix(seed=seed)
        q = torch.cat(quantiles).numpy()
        preds.append(q[:, q.shape[1] // 2])
        torch.cuda.empty_cache()
    bar.close()
    return np.mean(preds, axis=0)

if "kumo" in RUN_MODELS:
    cross_validate("kumo", kumo_fit_predict)

kumo:   0%|          | 0/18 [00:00<?, ?chunk/s]

kumo fold 0: MASE=0.3201


kumo:   0%|          | 0/18 [00:00<?, ?chunk/s]

kumo fold 1: MASE=0.3613


kumo:   0%|          | 0/18 [00:00<?, ?chunk/s]

...

kumo:   0%|          | 0/18 [00:00<?, ?chunk/s]

kumo fold 3: MASE=0.3311


kumo:   0%|          | 0/18 [00:00<?, ?chunk/s]

kumo fold 4: MASE=0.4101
kumo CV MASE: 0.3250


kumo:   0%|          | 0/111 [00:00<?, ?chunk/s]

### Neural network
MLP kecil dengan embedding untuk bioskop dan kolom kategori. Embedding memberi setiap bioskop dan kategori representasi yang dipelajari, sehingga bioskop dengan perilaku mirip saling berdekatan, sesuatu yang tidak bisa ditangkap tree dan Kumo secara langsung. Dilatih dengan weighted L1 loss yang sama dengan metrik (baris tambahan berbobot lebih rendah), lima seed dirata-rata, dan input numerik dipotong pada persentil ekstrem lalu distandarisasi. Kesalahannya berbeda dari LGBM dan Kumo, sehingga melengkapi blend.

In [74]:
NN_SEEDS = [1, 2, 3, 4, 5]
NN_EPOCHS = 15
NN_WIDTH, NN_DEPTH, NN_DROPOUT = 512, 4, 0.287
NN_LR, NN_WEIGHT_DECAY, NN_BATCH, NN_EMB_CAP = 7.9e-3, 6.9e-6, 256, 8
NN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class EmbeddingMLP(torch.nn.Module):
    def __init__(self, n_num, cardinalities, width=NN_WIDTH, depth=NN_DEPTH, dropout=NN_DROPOUT):
        super().__init__()
        self.emb = torch.nn.ModuleList([torch.nn.Embedding(c, min(NN_EMB_CAP, (c + 1) // 2 + 1)) for c in cardinalities])
        dim = n_num + sum(e.embedding_dim for e in self.emb)
        layers = []
        for k in range(depth):
            nxt = width if k == 0 else width // 2
            layers += [torch.nn.Linear(dim, nxt), torch.nn.BatchNorm1d(nxt), torch.nn.SiLU(), torch.nn.Dropout(dropout)]
            dim = nxt
        self.mlp = torch.nn.Sequential(*layers, torch.nn.Linear(dim, 1))

    def forward(self, x_num, x_cat):
        return self.mlp(torch.cat([x_num] + [e(x_cat[:, k]) for k, e in enumerate(self.emb)], dim=1)).squeeze(1)

def nn_prepare(X, st):
    v = X[st["num"]].astype(float)
    z = ((v.fillna(st["med"]).clip(st["lo"], st["hi"], axis=1) - st["mu"]) / st["sd"]).values
    flags = v[st["nan_cols"]].isna().astype(float).values
    codes = np.stack([X[c].cat.codes.values + 1 for c in CAT_COLS] + [X["cinema_code"].values + 1], axis=1)
    return (torch.tensor(np.hstack([z, flags]).astype(np.float32), device=NN_DEVICE), torch.tensor(codes.astype(np.int64), device=NN_DEVICE))

def train_nn(X_tr, y_tr, is_extra):
    num = [c for c in FEATURES if c not in CAT_COLS]
    ref = X_tr[num].astype(float)
    med, lo, hi = ref.median(), ref.quantile(0.01), ref.quantile(0.99)
    clipped = ref.clip(lo, hi, axis=1)
    st = {"num": num, "med": med, "lo": lo, "hi": hi, "mu": clipped.mean(), "sd": clipped.std() + 1e-9,
          "nan_cols": [c for c in num if ref[c].isna().mean() > 0.01],
          "cards": [len(X_tr[c].cat.categories) + 2 for c in CAT_COLS] + [len(CINEMA_CODE) + 2]}
    xn, xc = nn_prepare(X_tr, st)
    target = torch.tensor((y_tr + 0.05).astype(np.float32), device=NN_DEVICE)
    weight = torch.tensor(extra_weights(is_extra).astype(np.float32), device=NN_DEVICE)
    states = []
    for seed in tqdm(NN_SEEDS, desc="nn seeds", leave=False):
        torch.manual_seed(seed)
        net = EmbeddingMLP(xn.shape[1], st["cards"]).to(NN_DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=NN_LR / 1.5, weight_decay=NN_WEIGHT_DECAY)
        steps = NN_EPOCHS * int(np.ceil(len(xn) / NN_BATCH))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=NN_LR, total_steps=steps)
        for epoch in range(NN_EPOCHS):
            net.train()
            order = torch.randperm(len(xn), device=NN_DEVICE)
            for start in range(0, len(xn), NN_BATCH):
                idx = order[start:start + NN_BATCH]
                if len(idx) < 2:
                    continue
                loss = (weight[idx] * (net(xn[idx], xc[idx]) - target[idx]).abs()).mean()
                opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        states.append({k: v.detach().cpu() for k, v in net.state_dict().items()})
    return {"stats": st, "states": states}

def predict_nn(model, X_pred):
    xn, xc = nn_prepare(X_pred, model["stats"])
    preds = []
    for state in model["states"]:
        net = EmbeddingMLP(xn.shape[1], model["stats"]["cards"]).to(NN_DEVICE)
        net.load_state_dict(state)
        net.eval()
        with torch.no_grad():
            preds.append(net(xn, xc).cpu().numpy())
    return np.mean(preds, axis=0) - 0.05

def nn_fit_predict(X_tr, y_tr, is_extra, X_pred):
    return predict_nn(train_nn(X_tr, y_tr, is_extra), X_pred)

if "nn" in RUN_MODELS:
    cross_validate("nn", nn_fit_predict)

nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

nn fold 0: MASE=0.3671


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

nn fold 1: MASE=0.3850


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

...

nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

nn fold 3: MASE=0.3602


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

nn fold 4: MASE=0.4004
nn CV MASE: 0.3494


nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

### Perbandingan model
Cross-validation error dari setiap model yang dijalankan, per fold dan keseluruhan.

In [75]:
comparison = pd.DataFrame(
    {name: [mase_ratio(y, OOF[name])] + SCORES[name] for name in OOF},
    index=["CV"] + [f"fold {k}" for k in range(N_FOLDS)],
).T.sort_values("CV")
comparison.round(4)

,CV,fold 0,fold 1,fold 2,fold 3,fold 4
kumo,0.3250,0.3201,0.3613,0.2025,0.3311,0.4101
lgbm,0.3301,0.3313,0.3533,0.2117,0.3465,0.4081
nn,0.3494,0.3671,0.3850,0.2345,0.3602,0.4004


### Blend
Rata-rata berbobot dari tiga model: Kumo, LGBM, dan neural network. Ketiganya membuat kesalahan yang berbeda, sehingga rata-ratanya lebih stabil daripada salah satunya, dan Kumo memegang bobot terbesar. Nilai di bawah BLEND_SNAP dijadikan 0 dengan alasan yang sama seperti pada masing-masing model.

In [76]:
BLEND_WEIGHTS = {"kumo": 0.4, "lgbm": 0.3, "nn": 0.3}
BLEND_SNAP = 0.1
used = {m: w for m, w in BLEND_WEIGHTS.items() if m in OOF and w > 0}
total = sum(used.values())
snap_blend = lambda p: np.where(p < BLEND_SNAP, 0.0, p)
oof_blend = snap_blend(sum(w * OOF[m] for m, w in used.items()) / total)
test_blend = snap_blend(sum(w * TEST_PRED[m] for m, w in used.items()) / total)
print(f"blend {({m: round(w / total, 3) for m, w in used.items()})}: CV MASE {mase_ratio(y, oof_blend):.4f}")
train_df.assign(err=np.abs(y - oof_blend)).groupby("horizon")["err"].mean().round(4)

blend {'kumo': 0.4, 'lgbm': 0.3, 'nn': 0.3}: CV MASE 0.3251


horizon
4     0.3424
5     0.3569
6     0.3260
7     0.3228
8     0.2907
9     0.3147
10    0.3219
Name: err, dtype: float64

# Postprocessing
Model dilatih hanya pada April sampai September, sehingga tiga efek musim test yang tidak bisa ditunjukkan oleh data train ditangani di sini.

### Level musim dari pasar nasional tahun lalu
Model belum pernah melihat Ramadan atau Lebaran. Untuk baris tersebut, level berasal dari rumus yang hanya memakai chart nasional musim lalu (dipublikasikan sampai 2025-09-30): rasio nasional D_h / D1-D3 yang diharapkan adalah perubahan pasar nasional antara hari-hari yang sama pada musim lalu, pada jarak yang sama dari Idulfitri, dikali perubahan pangsa pasar film. Efek hari dan hari libur yang biasa dari musim lalu diganti dengan milik 2026 dan pasar dihaluskan selama 3 hari. Prediksi model dikonversi ke skala ini dengan q(h), yaitu prediksi model untuk film train dibagi rasio nasionalnya.
- ram_to_lebaran: level ditetapkan per horizon, karena model memprediksi penurunan pada minggu yang belum pernah dilihatnya, dan LGBM memberikan peringkat pasangan.
- ram_to_ram, pre_to_ram: satu faktor level per kelompok, bentuk harian milik model dipertahankan.
- Baris normal dan xmas mempertahankan level model.

In [77]:
USE_SEASON_FORMULA = True
LEBARAN_MODEL = "lgbm"
FORMULA_GROUPS = ["ram_to_lebaran", "ram_to_ram", "pre_to_ram"]
MARKET_FILES = ["cinepoint_daily_top_prev.csv", "cinepoint_daily_top_train.csv"]
EID_2026, EID_2025 = pd.Timestamp("2026-03-21"), pd.Timestamp("2025-03-31")
HOLIDAYS_2024_25 = pd.to_datetime(["2024-11-27", "2024-12-25", "2024-12-26", "2025-01-01", "2025-01-27", "2025-01-28",
                                   "2025-01-29", "2025-03-28"])
RAMADAN_START, LEBARAN_START = ramadan["date"].min(), pd.Timestamp("2026-03-20")
CP_FORMAT = r"\s*\((?:IMAX[^)]*|3D|2D|4DX)\)\s*$"
cp_key = lambda s: re.sub(r"[^a-z0-9]", "", re.sub(CP_FORMAT, "", s, flags=re.I).lower())
HZ = list(range(4, 11))

def season_group(df):
    d13_ramadan = np.any([((df["d1_date"] + pd.Timedelta(days=k)) >= RAMADAN_START) & ((df["d1_date"] + pd.Timedelta(days=k)) < LEBARAN_START) for k in range(3)], axis=0)
    in_ramadan = (df["date_show"] >= RAMADAN_START) & (df["date_show"] < LEBARAN_START)
    xmas = (df["date_show"] >= "2025-12-20") & (df["date_show"] <= "2026-01-04")
    return np.select([d13_ramadan & ~in_ramadan, d13_ramadan, in_ramadan, xmas],
                     ["ram_to_lebaran", "ram_to_ram", "pre_to_ram", "xmas"], "normal")

def replace_chart_dates(charts, fix_files):
    """a few dates of the daily Cinepoint files carry another day's numbers (a film's running total does not equal the day before plus that day's admissions); the re-scraped rows replace those dates"""
    files = [f for f in fix_files if os.path.exists(f)]
    if not files:
        print("WARNING: no cinepoint_daily_fix_*.csv found, the mislabelled chart dates are NOT corrected (upload the fix files next to the daily files)")
        return charts
    fix = pd.concat([pd.read_csv(f, parse_dates=["date"]) for f in files], ignore_index=True)
    fix = fix[(fix["date"] >= charts["date"].min()) & (fix["date"] <= charts["date"].max())]
    print(f"chart dates replaced by the re-scraped rows: {fix['date'].nunique()} ({len(files)} fix file(s))")
    return pd.concat([charts[~charts["date"].isin(fix["date"].unique())], fix], ignore_index=True)

def load_charts():
    cpd = pd.concat([pd.read_csv(f"{EXTERNAL_DIR}/cinepoint/{n}", parse_dates=["date"]) for n in MARKET_FILES], ignore_index=True)
    cpd = replace_chart_dates(cpd, [f"{EXTERNAL_DIR}/cinepoint/cinepoint_daily_fix_pre_cutoff.csv"])
    assert cpd["date"].max() <= pd.Timestamp("2025-09-30"), "post-cutoff Cinepoint data in the market files"
    cpd = cpd.drop_duplicates(["date", "rank"]).assign(key=lambda d: d["title"].map(cp_key))
    market = cpd.groupby("date")["daily_admission"].sum().loc["2024-09-20":]
    return cpd.groupby(["key", "date"])["daily_admission"].sum(), market, cpd.groupby("date")["daily_admission"].min()

def calendar_effect(market):
    """usual weekday effect and weekday-holiday effect on the market (log vs its 15-day median, normal weeks only)"""
    idx = market.index
    off_days = pd.DatetimeIndex(HOLIDAYS_2024_25).union(pd.DatetimeIndex(NATIONAL_HOLIDAYS)).union(pd.DatetimeIndex(cuti["date"]))
    dev = np.log(market) - np.log(market).rolling(15, center=True, min_periods=8).median()
    weekday_holiday = idx.isin(off_days) & (idx.dayofweek < 5)
    normal_weeks = ~(((idx >= "2024-12-18") & (idx <= "2025-01-06")) | ((idx >= "2025-02-10") & (idx <= "2025-04-20"))) & dev.notna().values
    dow_eff = dev[normal_weeks & ~weekday_holiday].groupby(idx[normal_weeks & ~weekday_holiday].dayofweek).mean()
    hol_eff = (dev[normal_weeks & weekday_holiday] - dow_eff.reindex(idx[normal_weeks & weekday_holiday].dayofweek).values).mean()
    return lambda d: np.exp(dow_eff.reindex(pd.DatetimeIndex(d).dayofweek).values
                            + hol_eff * (pd.DatetimeIndex(d).isin(off_days) & (pd.DatetimeIndex(d).dayofweek < 5)))

def share_change(daily, market, chart_min, d1_from, d1_to, min_d13):
    """films' market share on D4-D10 / their share on D1-D3 (size-weighted geometric mean), films with D1 in [d1_from, d1_to]"""
    first = daily.reset_index().groupby("key")["date"].min()
    rows, weights = [], []
    for key, d1 in first[(first >= d1_from) & (first <= d1_to)].items():
        days = pd.date_range(d1, periods=10)
        a = daily[key].reindex(days)
        if a.iloc[:3].isna().any() or a.iloc[:3].mean() < min_d13:
            continue
        share = a.fillna(0.7 * chart_min.reindex(days)) / market.reindex(days).values
        rows.append(np.log(np.clip(share.iloc[3:].values / share.iloc[:3].mean(), 1e-3, None)))
        weights.append(np.sqrt(a.iloc[:3].mean()))
    return pd.Series(np.exp(np.average(rows, axis=0, weights=weights)), index=HZ)

def model_scale(oof, daily):
    """q(h): the model's mean prediction for a train film / that film's national D_h / D1-D3 ratio (median over films)"""
    tr = train_df.assign(key=train_df["movie_title"].map(cp_key), p=oof)
    d1 = tr.groupby("key")["d1_date"].min()
    pred = tr.groupby(["key", "horizon"])["p"].mean().unstack()
    rows = []
    for key in pred.index.intersection(daily.index.get_level_values(0).unique()):
        s = daily[key].reindex(pd.date_range(d1[key], periods=10))
        if s.iloc[:3].isna().any() or s.iloc[3:].isna().any() or s.iloc[:3].mean() < 3000:
            continue
        rows.append(np.log(pred.loc[key, HZ].values.clip(1e-3)) - np.log(s.iloc[3:].values / s.iloc[:3].mean()))
    return pd.Series(np.exp(np.nanmedian(rows, axis=0)), index=HZ)

def national_ratio(df, market, calendar, share_lebaran, share_normal, lebaran):
    """expected national D_h / D1-D3: market change between the same days of last season x share change"""
    seasonal = (market / calendar(market.index)).rolling(3, center=True, min_periods=1).mean()
    est = lambda d: seasonal.reindex(pd.DatetimeIndex(d) - (EID_2026 - EID_2025)).values * calendar(d)
    market_change = est(df["date_show"]) / np.mean([est(df["d1_date"] + pd.Timedelta(days=k)) for k in range(3)], axis=0)
    h = df["horizon"].values
    return market_change * np.where(lebaran, share_lebaran.reindex(h).values, share_normal.reindex(h).values)

test_submit = test_blend.copy()
if USE_SEASON_FORMULA:
    daily, market, chart_min = load_charts()
    calendar = calendar_effect(market)
    share_lebaran = share_change(daily, market, chart_min, EID_2025 - pd.Timedelta(days=1), EID_2025 + pd.Timedelta(days=1), 20000)
    share_normal = share_change(daily, market, chart_min, pd.Timestamp("2025-04-01"), pd.Timestamp("2025-09-19"), 3000)
    season_groups = season_group(test_df)
    rows = np.isin(season_groups, FORMULA_GROUPS)
    sub = test_df[rows]
    nat = national_ratio(sub, market, calendar, share_lebaran, share_normal, season_groups[rows] == "ram_to_lebaran")
    lebaran_model = LEBARAN_MODEL if LEBARAN_MODEL in TEST_PRED else None
    summary = []
    for g in FORMULA_GROUPS:
        m = season_groups[rows] == g
        if not m.any():
            continue
        on_lgbm = g == "ram_to_lebaran" and lebaran_model is not None
        pred = (TEST_PRED[lebaran_model] if on_lgbm else test_blend)[rows][m]
        target = nat[m] * model_scale(OOF[lebaran_model] if on_lgbm else oof_blend, daily).reindex(sub["horizon"].values[m]).values
        h = sub["horizon"].values[m]
        if g == "ram_to_lebaran":
            factor = pd.Series(target).groupby(h).transform("mean").values / pd.Series(pred).groupby(h).transform("mean").clip(lower=1e-9).values
        else:
            factor = np.full(m.sum(), target.mean() / max(pred.mean(), 1e-9))
        idx = np.flatnonzero(rows)[m]
        test_submit[idx] = pred * factor
        summary.append({"group": g, "rows": int(m.sum()), "model mean": pred.mean(), "formula mean": target.mean(), "factor": np.mean(factor)})
    print(f"share change D4-D10, 2025 Eid releases: {share_lebaran.round(2).tolist()} | normal films: {share_normal.round(2).tolist()}")
    display(pd.DataFrame(summary).round(3))

chart dates replaced by the re-scraped rows: 36 (1 fix file(s))
share change D4-D10, 2025 Eid releases: [1.07, 1.07, 1.06, 1.05, 1.11, 1.09, 1.0] | normal films: [0.77, 0.78, 0.73, 0.52, 0.27, 0.2, 0.18]


,group,rows,model mean,formula mean,factor
0,ram_to_lebaran,4598,0.533,2.205,4.857
1,ram_to_ram,10767,0.347,0.422,1.215
2,pre_to_ram,1962,0.165,0.153,0.929


### Faktor genre Ramadan
Pada Ramadan tahun-tahun sebelumnya, diukur dari chart nasional sampai 2025-09-30, genre tidak bertahan sama: film action, drama, dan keluarga bertahan lebih baik dari minggu biasa dan film horor bertahan lebih buruk. Sebagian besar film Ramadan masa lalu adalah horor, sehingga film Ramadan yang khas pada masa itu menurun seperti film horor, sedangkan film Ramadan pada periode test sebagian besar bukan horor. Baris dengan hari target di Ramadan mendapat faktor sesuai genrenya. Rumus musim sudah menetapkan level tiap kelompok Ramadan, sehingga faktor ini hanya menggeser genre di dalam level tersebut.

In [78]:
USE_RAMADAN_GENRE = True
RAMADAN_GENRE_FACTORS = {"action": 1.27, "drama": 1.37, "family": 1.48, "horror": 0.80}
ACTION_GENRES = ["Action", "Sci-Fi", "Superhero", "Adventure", "Thriller", "Crime", "War", "Fantasy", "Disaster"]
DRAMA_GENRES = ["Drama", "Romance", "Biography", "History"]

def genre_group(df):
    """one coarse group per row, in this order: anime, family, comedy, horror, action, drama, other"""
    has = lambda names: np.any([df[f"genre_{g}"].values == 1 for g in names if f"genre_{g}" in df], axis=0)
    return np.select([df["kind_code"].values == 2, has(["Animation", "Family"]), has(["Comedy"]), has(["Horror"]), has(ACTION_GENRES), has(DRAMA_GENRES)],
                     ["anime", "family", "comedy", "horror", "action", "drama"], "other")

in_ramadan = ((test_df["date_show"] >= RAMADAN_START) & (test_df["date_show"] < LEBARAN_START)).values

if USE_RAMADAN_GENRE:
    gf = pd.Series(genre_group(test_df)).map(RAMADAN_GENRE_FACTORS).fillna(1.0).values
    season_groups = season_group(test_df)
    for g in ["pre_to_ram", "ram_to_ram"]:
        m = (season_groups == g) & in_ramadan
        new = test_submit[m] * gf[m]
        test_submit[m] = new * test_submit[m].mean() / new.mean()
    print("Ramadan rows by genre group:", pd.Series(genre_group(test_df)[in_ramadan]).value_counts().to_dict())
test_df["total_ticket"] = test_submit * test_df["scale"]
test_df["total_ticket"].sum()

Ramadan rows by genre group: {'action': 5266, 'drama': 2676, 'family': 2288, 'comedy': 1229, 'horror': 1046, 'other': 224}


np.float64(12778205.043974655)

### Lift horizon akhir dan Ramadan
Model dilatih pada April sampai September, ketika film menurun lebih cepat daripada musim test, sehingga model memprediksi terlalu rendah pada hari-hari akhir masa tayang film. Horizon akhir dan kelompok Ramadan dikali dengan pengali tetap (LATE_LIFT dan RAMADAN_LIFT) yang dikalibrasi dari umpan balik submission.

In [79]:
USE_LIFT = True
LATE_FROM, LATE_LIFT = 7, 1.2
RAMADAN_LIFT = 1.1

if USE_LIFT:
    lift = np.where(test_df["horizon"].values >= LATE_FROM, LATE_LIFT, 1.0) * np.where(np.isin(season_group(test_df), ["ram_to_ram", "pre_to_ram"]), RAMADAN_LIFT, 1.0)
    test_submit = test_submit * lift
test_df["total_ticket"] = test_submit * test_df["scale"]
test_df["total_ticket"].sum()

np.float64(13946362.54260886)

# Submission

In [80]:
def write_submission(ratio, path):
    out = test_df[["id"]].astype({"id": int}).assign(total_ticket=ratio * test_df["scale"].values)
    sub = sample_sub[["id"]].merge(out, on="id", how="left")
    assert len(sub) == len(sample_sub) and sub["total_ticket"].notna().all()
    sub.to_csv(path, index=False)
    return sub

submission = write_submission(test_submit, "submission.csv")
submission.describe()

,id,total_ticket
count,72611.000000,72611.000000
mean,36306.000000,192.069556
std,20961.134535,636.989975
min,1.000000,0.000000
25%,18153.500000,0.000000
50%,36306.000000,30.586992
75%,54458.500000,156.662024
max,72611.000000,31177.220632


### Ekspor model
Menyimpan bagian model yang dilatih dalam satu file: model LGBM dan neural network yang di-fit pada semua baris train dan baris tambahan, baris dan seed dari konteks Kumo, dan semua pengaturan blend, langkah musim, dan lift. Kumo-Tabular adalah model pretrained sehingga bobotnya tidak disimpan; bobotnya diunduh oleh cell bobot Kumo di atas. Setelah disimpan, file dimuat ulang dan harus menghasilkan prediksi test LGBM dan neural network yang sama.

In [81]:
MODEL_PATH = "dikeri_ivant_model.joblib"
X_fit, y_fit, is_extra_fit = fit_rows(np.arange(len(X)))
lgbm_models = [LGBMRegressor(objective="l1", random_state=seed, verbose=-1, **LGBM_PARAMS).fit(X_fit[FEATURES], y_fit, sample_weight=extra_weights(is_extra_fit)) for seed in MODEL_SEEDS] if "lgbm" in RUN_MODELS else []
kumo_context = []
if "kumo" in RUN_MODELS:
    real_idx, extra_idx = np.flatnonzero(~is_extra_fit), np.flatnonzero(is_extra_fit)
    for seed in KUMO_CONTEXT_SEEDS:
        rng = np.random.default_rng(seed)
        ctx = np.r_[rng.choice(real_idx, min(KUMO_CONTEXT_ROWS, len(real_idx)), replace=False), rng.choice(extra_idx, min(KUMO_EXTRA_ROWS, len(extra_idx)), replace=False)].astype(int)
        kumo_context.append({"seed": seed, "X": X_fit.iloc[ctx][FEATURES].reset_index(drop=True), "y": y_fit[ctx]})
nn_model = train_nn(X_fit, y_fit, is_extra_fit) if "nn" in RUN_MODELS else None
nn_test_before_saving = predict_nn(nn_model, X_test) if nn_model is not None else None
season_and_lift_names = ["USE_SEASON_FORMULA", "LEBARAN_MODEL", "FORMULA_GROUPS", "MARKET_FILES", "EID_2026", "EID_2025", "RAMADAN_START", "LEBARAN_START", "USE_RAMADAN_GENRE", "RAMADAN_GENRE_FACTORS", "USE_LIFT", "LATE_FROM", "LATE_LIFT", "RAMADAN_LIFT"]
joblib.dump({
    "features": FEATURES, "cat_cols": CAT_COLS, "cat_levels": {c: list(v) for c, v in CAT_LEVELS.items()},
    "lgbm_models": lgbm_models, "lgbm_params": LGBM_PARAMS, "model_seeds": MODEL_SEEDS, "extra_weight": EXTRA_WEIGHT, "zero_snap": ZERO_SNAP,
    "nn_model": nn_model, "nn_settings": {"seeds": NN_SEEDS, "epochs": NN_EPOCHS, "width": NN_WIDTH, "depth": NN_DEPTH, "dropout": NN_DROPOUT, "lr": NN_LR, "weight_decay": NN_WEIGHT_DECAY, "batch": NN_BATCH, "emb_cap": NN_EMB_CAP}, "cinema_code": CINEMA_CODE,
    "kumo_repo": KUMO_REPO, "kumo_size": "large", "kumo_context": kumo_context, "kumo_estimators": KUMO_ESTIMATORS, "kumo_stypes": KUMO_STYPES,
    "blend_weights": BLEND_WEIGHTS, "blend_snap": BLEND_SNAP, "season_and_lift": {n: globals().get(n) for n in season_and_lift_names},
}, MODEL_PATH, compress=3)
print(f"saved {MODEL_PATH}: {os.path.getsize(MODEL_PATH) / 1e6:.1f} MB | {len(lgbm_models)} LGBM models | {len(kumo_context)} Kumo contexts | NN {'yes' if nn_model is not None else 'no'}")
if lgbm_models:
    saved = joblib.load(MODEL_PATH)
    check = np.mean([m.predict(X_test[saved["features"]]) for m in saved["lgbm_models"]], axis=0)
    check = np.where(np.clip(check, 0, None) < saved["zero_snap"]["lgbm"], 0.0, np.clip(check, 0, None))
    print("reloaded LGBM models reproduce the test predictions:", bool(np.allclose(check, TEST_PRED["lgbm"], atol=1e-9)))
if nn_model is not None:
    saved = joblib.load(MODEL_PATH)
    print("reloaded NN reproduces its predictions:", bool(np.allclose(predict_nn(saved["nn_model"], X_test), nn_test_before_saving, atol=1e-5)))

nn seeds:   0%|          | 0/5 [00:00<?, ?it/s]

saved dikeri_ivant_model.joblib: 26.2 MB | 3 LGBM models | 3 Kumo contexts | NN yes
reloaded LGBM models reproduce the test predictions: True
reloaded NN reproduces its predictions: True
